# BTL02 - FER2013 CNN (PyTorch)

Cả nhóm dùng chung notebook này. **Mỗi người chạy trên Kaggle của mình, chỉ đổi biến `NGUOI` ở ô "Cấu hình"** (An / Quyen / Thanh / Tan). Mỗi người chỉ chạy phần việc của mình. **Tấn** (`NGUOI = "Tan"`) chạy khám phá dữ liệu (histogram, ảnh mẫu), không cần GPU, rồi tải ảnh ở tab Output.

**Giai đoạn 1 (song song, mỗi người một tài khoản Kaggle):**
1. Import notebook, Add Input bộ FER-2013 (có `train/`, `test/`), bật GPU (Accelerator → GPU T4).
2. Ô "Cấu hình": đặt `NGUOI` = tên của bạn.
3. Chạy thử các ô đến hết "Chạy thử 1 epoch" xem có lỗi không.
4. **Save Version → Save & Run All (Commit)**. Kaggle chạy nền, có thể đóng trình duyệt.
5. Xong thì vào tab **Output**, tải file `runs_ketqua_<tên>.zip` và gửi cho An và Thanh.

**Giai đoạn 2 (chỉ An và Thanh, sau khi mọi người đã xong):**
1. Upload các file zip của cả nhóm (kể cả zip của chính mình) lên Kaggle: Add Input → Upload → New Dataset (kéo thả các file `runs_ketqua_*.zip`).
2. Ô "Cấu hình": đặt `CHAY_THI_NGHIEM = False`.
3. Save & Run All. An sẽ ra bảng tổng hợp `results.csv`, Thanh ra confusion matrix, ảnh đúng/sai và filter của model tốt nhất.

In [1]:
import argparse
import csv
import glob
import json
import math
import os
import random
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

SPLITS = ["Training", "PublicTest", "PrivateTest"]  # train / validation / test
CLASSES_CSV = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]


# ----------------------------------------------------------------------------
# Du lieu
# ----------------------------------------------------------------------------
def _find_dir(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None



def _read_folder(split_dir, classes):
    xs, ys = [], []

    for ci, c in enumerate(classes):
        cdir = os.path.join(split_dir, c)
        files = sorted(os.listdir(cdir))
        count = 0

        for fn in files:
            if fn.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(cdir, fn)

                try:
                    with Image.open(path) as im:
                        img = im.convert("L").resize(
                            (48, 48), Image.Resampling.BILINEAR
                        )
                        xs.append(np.asarray(img, dtype=np.uint8))
                        ys.append(ci)
                        count += 1

                except Exception as e:
                    print(f"Loi anh: {path} | {e}")

        print(f"Da doc lop {c}: {count} anh")

    return np.stack(xs), np.array(ys, dtype=np.int64)



def _find_csv(data_dir):
    names = ["fer2013.csv", "icml_face_data.csv"] + [
        os.path.basename(p) for p in sorted(glob.glob(os.path.join(data_dir, "*.csv")))]
    for n in names:
        p = os.path.join(data_dir, n)
        if os.path.exists(p):
            with open(p, newline="") as f:
                head = f.readline()
            if "pixels" in head:
                return p
    return None


def load_data(data_dir):
    """Tra ve ({split: (x uint8 NxHxW, y int N)}, danh sach ten lop).
    Ho tro: file CSV Kaggle (fer2013.csv / icml_face_data.csv), thu muc
    Training+PublicTest+PrivateTest, hoac thu muc train+test (khi do chia doi thu muc test thanh validation va test)."""
    caches = [os.path.join(data_dir, "fer_cache.npz"), "fer_cache.npz"]
    for cache in caches:
        if os.path.exists(cache):
            z = np.load(cache, allow_pickle=True)
            return {s: (z[s + "_x"], z[s + "_y"]) for s in SPLITS}, [str(c) for c in z["classes"]]

    csv_path = _find_csv(data_dir) if os.path.isdir(data_dir) else None
    if csv_path:
        xs = {s: [] for s in SPLITS}
        ys = {s: [] for s in SPLITS}
        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            reader.fieldnames = [n.strip() for n in reader.fieldnames]  # file icml co khoang trang thua
            for row in reader:
                s = row["Usage"].strip()
                xs[s].append(np.array(row["pixels"].split(), dtype=np.uint8).reshape(48, 48))
                ys[s].append(int(row["emotion"]))
        classes = CLASSES_CSV
        data = {s: (np.stack(xs[s]), np.array(ys[s], dtype=np.int64)) for s in SPLITS}
    else:
        dirs = [_find_dir(data_dir, [s]) for s in SPLITS]
        if all(dirs):
            classes = sorted(d for d in os.listdir(dirs[0]) if os.path.isdir(os.path.join(dirs[0], d)))
            data = {s: _read_folder(d, classes) for s, d in zip(SPLITS, dirs)}
        else:
            tr_dir = _find_dir(data_dir, ["train", "Train"])
            te_dir = _find_dir(data_dir, ["test", "Test"])
            if not (tr_dir and te_dir):
                here = sorted(os.listdir(data_dir)) if os.path.isdir(data_dir) else "(thu muc khong ton tai)"
                raise FileNotFoundError(
                    f"Khong tim thay du lieu trong '{os.path.abspath(data_dir)}'.\n"
                    f"Thu muc dang chay: {os.getcwd()}\nNoi dung thu muc data: {here}\n"
                    "Can 1 trong 3: file fer2013.csv (hoac icml_face_data.csv); "
                    "thu muc Training/PublicTest/PrivateTest; hoac thu muc train/test. "
                    "Neu du lieu o cho khac, doi DATA_DIR thanh duong dan day du.")
            classes = sorted(d for d in os.listdir(tr_dir) if os.path.isdir(os.path.join(tr_dir, d)))
            x, y = _read_folder(tr_dir, classes)
            xt, yt = _read_folder(te_dir, classes)
            perm = np.random.RandomState(0).permutation(len(xt))
            half = len(xt) // 2
            vi, ti = perm[:half], perm[half:]
            data = {"Training": (x, y), "PublicTest": (xt[vi], yt[vi]), "PrivateTest": (xt[ti], yt[ti])}
            print(f"Luu y: thu muc test ({len(xt)} anh) duoc chia ngau nhien thanh 2 nua: "
                  f"validation ({len(vi)}) va test ({len(ti)}).")

    for cache in caches:  # luu cache de cac lan chay sau nhanh hon
        try:
            np.savez(cache, classes=np.array(classes),
                     **{s + "_x": data[s][0] for s in SPLITS},
                     **{s + "_y": data[s][1] for s in SPLITS})
            break
        except OSError:
            continue
    return data, classes


class FER(Dataset):
    def __init__(self, x, y, train=False, aug=True):
        self.x, self.y = x, y
        tf = []
        if train and aug:
            tf += [T.RandomHorizontalFlip(),
                   T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1))]
        tf += [T.ToTensor(), T.Normalize((0.5,), (0.5,))]
        self.tf = T.Compose(tf)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.x[i])), int(self.y[i])


def make_loaders(data, a):
    tr = FER(*data["Training"], train=True, aug=bool(a.aug))
    va = FER(*data["PublicTest"])
    te = FER(*data["PrivateTest"])
    kw = dict(batch_size=a.batch, num_workers=a.workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, shuffle=True, **kw),
            DataLoader(va, shuffle=False, **kw),
            DataLoader(te, shuffle=False, **kw))


# ----------------------------------------------------------------------------
# Mo hinh: `blocks` khoi, moi khoi gom `convs` lop Conv3x3 + MaxPool + Dropout
# so kenh: base, 2*base, 4*base, ... (toi da 512)
# ----------------------------------------------------------------------------
class CNN(nn.Module):
    def __init__(self, n_classes, blocks=3, convs=2, base=64, fc=1024, dropout=0.2, bn=True):
        super().__init__()
        if not 1 <= blocks <= 5:
            raise ValueError("--blocks phai tu 1 den 5 (anh 48x48 chi pool duoc toi da 5 lan)")
        layers, cin = [], 1
        for b in range(blocks):
            cout = min(base * (2 ** b), 512)
            for _ in range(convs):
                layers.append(nn.Conv2d(cin, cout, 3, padding=1))
                if bn:
                    layers.append(nn.BatchNorm2d(cout))
                layers.append(nn.ReLU(inplace=True))
                cin = cout
            layers += [nn.MaxPool2d(2), nn.Dropout(dropout)]
        self.features = nn.Sequential(*layers)
        size = 48 // (2 ** blocks)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(cin * size * size, fc), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(fc, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(cfg, n_classes):
    return CNN(n_classes, cfg["blocks"], cfg["convs"], cfg["base"], cfg["fc"],
               cfg["dropout"], bool(cfg["bn"]))


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def run_eval(model, loader, dev, crit=None):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        out = model(xb)
        if crit is not None:
            total_loss += crit(out, yb).item() * len(yb)
        preds.append(out.argmax(1).cpu().numpy())
        labels.append(yb.cpu().numpy())
    preds, labels = np.concatenate(preds), np.concatenate(labels)
    return total_loss / len(labels), float((preds == labels).mean()), preds, labels


def load_run(a):
    """Doc checkpoint tot nhat cua thi nghiem --name."""
    ck = torch.load(os.path.join(a.runs, a.name, "best.pt"), map_location="cpu", weights_only=False)
    model = build_model(ck["args"], len(ck["classes"]))
    model.load_state_dict(ck["model"])
    return model.to(get_device()).eval(), ck


# ----------------------------------------------------------------------------
# train
# ----------------------------------------------------------------------------
def cmd_train(a):
    random.seed(a.seed)
    np.random.seed(a.seed)
    torch.manual_seed(a.seed)
    dev = get_device()
    out_dir = os.path.join(a.runs, a.name)
    os.makedirs(out_dir, exist_ok=True)

    data, classes = load_data(a.data_dir)
    tr, va, te = make_loaders(data, a)
    print("Train/Val/Test:", len(tr.dataset), len(va.dataset), len(te.dataset),
          "| lop:", classes, "| thiet bi:", dev)

    cfg = {k: v for k, v in vars(a).items() if k != "fn"}   # bo ham fn de checkpoint luu duoc
    model = build_model(cfg, len(classes)).to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    print("So tham so:", n_params)

    weight = None
    if a.class_weight:
        counts = np.bincount(data["Training"][1], minlength=len(classes)).astype(np.float32)
        weight = torch.tensor(counts.sum() / (len(counts) * counts)).to(dev)
    crit = nn.CrossEntropyLoss(weight=weight)

    if a.opt == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=a.lr, weight_decay=a.wd)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=a.lr, momentum=0.9, weight_decay=a.wd)
    sched = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.1, patience=3)
             if a.sched else None)

    hist = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, bad, t0 = float("inf"), 0, 0, time.time()
    for ep in range(1, a.epochs + 1):
        model.train()
        tl, tc, n = 0.0, 0, 0
        for xb, yb in tr:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            tl += loss.item() * len(yb)
            tc += (out.argmax(1) == yb).sum().item()
            n += len(yb)
        vl, vacc, _, _ = run_eval(model, va, dev, crit)
        if sched:
            sched.step(vl)
        for k, v in zip(hist, [ep, tl / n, tc / n, vl, vacc]):
            hist[k].append(v)
        print(f"Epoch {ep:3d} | train loss {tl/n:.4f} acc {tc/n:.4f} | "
              f"val loss {vl:.4f} acc {vacc:.4f} | lr {opt.param_groups[0]['lr']:.1e}")

        if vl < best_loss:
            best_loss, best_epoch, bad = vl, ep, 0
            torch.save({"model": model.state_dict(), "args": cfg, "classes": classes},
                       os.path.join(out_dir, "best.pt"))
        else:
            bad += 1
            if bad >= a.patience:
                print(f"EarlyStopping: {a.patience} epoch khong cai thien val_loss.")
                break

    # log + duong cong
    with open(os.path.join(out_dir, "log.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(list(hist))
        w.writerows(zip(*hist.values()))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["epoch"], hist["train_acc"], label="train")
    ax[0].plot(hist["epoch"], hist["val_acc"], label="val")
    ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist["epoch"], hist["train_loss"], label="train")
    ax[1].plot(hist["epoch"], hist["val_loss"], label="val")
    ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.suptitle(f"{a.name} (best epoch {best_epoch})")
    fig.savefig(os.path.join(out_dir, "curves.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)

    # danh gia model tot nhat tren val + test (test chi dung 1 lan, sau khi train xong)
    ck = torch.load(os.path.join(out_dir, "best.pt"), map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"])
    vl, vacc, _, _ = run_eval(model, va, dev, crit)
    _, tacc, _, _ = run_eval(model, te, dev, crit)
    result = {"name": a.name, "best_epoch": best_epoch, "epochs_run": len(hist["epoch"]),
              "val_loss": round(vl, 4), "val_acc": round(vacc, 4), "test_acc": round(tacc, 4),
              "params": n_params, "minutes": round((time.time() - t0) / 60, 1),
              "config": {k: cfg[k] for k in ["blocks", "convs", "base", "fc", "dropout", "bn",
                                              "opt", "lr", "wd", "sched", "aug",
                                              "class_weight", "batch", "seed"]}}
    with open(os.path.join(out_dir, "result.json"), "w") as f:
        json.dump(result, f, indent=2)
    print("KET QUA:", json.dumps({k: v for k, v in result.items() if k != "config"}))


# ----------------------------------------------------------------------------
# eval
# ----------------------------------------------------------------------------
def save_grid(imgs, titles, path, title, rows=4, cols=8):
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9))
    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < len(imgs):
            ax.imshow(imgs[i], cmap="gray")
            ax.set_title(titles[i], fontsize=8)
    fig.suptitle(title)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def cmd_eval(a):
    dev = get_device()
    model, ck = load_run(a)
    classes = ck["classes"]
    out_dir = os.path.join(a.runs, a.name)
    data, _ = load_data(a.data_dir)
    x, y = data["PrivateTest"]
    _, acc, pred, lab = run_eval(model, DataLoader(FER(x, y), batch_size=256, num_workers=0), dev)

    k = len(classes)
    cm = np.zeros((k, k), dtype=int)
    for t, p in zip(lab, pred):
        cm[t, p] += 1
    pct = cm / cm.sum(1, keepdims=True).clip(min=1) * 100
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(k)); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(k)); ax.set_yticklabels(classes)
    for i in range(k):
        for j in range(k):
            ax.text(j, i, f"{pct[i, j]:.0f}%", ha="center", va="center",
                    color="white" if pct[i, j] > 50 else "black", fontsize=8)
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
    ax.set_title(f"Confusion matrix - test accuracy {acc * 100:.2f}%")
    fig.colorbar(im)
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)

    lines = [f"Test accuracy: {acc * 100:.2f}%", "", f"{'class':10s} precision recall   f1   support"]
    for c in range(k):
        tp = cm[c, c]
        prec = tp / max(cm[:, c].sum(), 1)
        rec = tp / max(cm[c].sum(), 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        lines.append(f"{classes[c]:10s} {prec:9.3f} {rec:6.3f} {f1:6.3f} {cm[c].sum():7d}")
    with open(os.path.join(out_dir, "report.txt"), "w") as f:
        f.write("\n".join(lines))
    print("\n".join(lines))

    rng = np.random.RandomState(0)
    for tag, idx in [("wrong", np.where(pred != lab)[0]), ("right", np.where(pred == lab)[0])]:
        idx = rng.choice(idx, size=min(32, len(idx)), replace=False)
        save_grid([x[i] for i in idx],
                  [f"{classes[pred[i]]} ({classes[lab[i]]})" for i in idx],
                  os.path.join(out_dir, f"{tag}_cases.png"),
                  f"{tag.capitalize()} predictions: Predict (True)")
    print("Da luu confusion_matrix.png, report.txt, wrong_cases.png, right_cases.png vao", out_dir)


# ----------------------------------------------------------------------------
# explore: kham pha du lieu (histogram so anh moi lop, anh mau) - khong can GPU
# ----------------------------------------------------------------------------
def cmd_explore(a):
    data, classes = load_data(a.data_dir)
    out_dir = os.path.join(a.runs, "data_explore")
    os.makedirs(out_dir, exist_ok=True)
    names = {"Training": "Train", "PublicTest": "Validation", "PrivateTest": "Test"}
    k = len(classes)
    counts = {s: np.bincount(data[s][1], minlength=k) for s in SPLITS}

    h, w = data["Training"][0].shape[1:3]
    lines = [f"Kich thuoc anh: {h} x {w} x 1 (anh xam)", "Cac lop: " + ", ".join(classes), "",
             f"{'lop':10s}" + "".join(f"{names[s]:>12s}" for s in SPLITS)]
    for c in range(k):
        lines.append(f"{classes[c]:10s}" + "".join(f"{counts[s][c]:12d}" for s in SPLITS))
    lines.append(f"{'TONG':10s}" + "".join(f"{counts[s].sum():12d}" for s in SPLITS))
    with open(os.path.join(out_dir, "data_info.txt"), "w") as f:
        f.write("\n".join(lines))
    print("\n".join(lines))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, s in zip(axes, SPLITS):
        ax.bar(classes, counts[s])
        ax.set_title(f"Histogram - {names[s]} ({counts[s].sum()} anh)")
        ax.tick_params(axis="x", rotation=45)
    fig.savefig(os.path.join(out_dir, "histogram.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)

    x, y = data["Training"]
    idx = np.random.RandomState(0).choice(len(x), 32, replace=False)
    save_grid([x[i] for i in idx], [classes[y[i]] for i in idx],
              os.path.join(out_dir, "samples.png"), "Anh mau tap train")
    print("Da luu histogram.png, samples.png, data_info.txt vao", out_dir)


# ----------------------------------------------------------------------------
# filters: ve trong so cua lop Conv + feature map cua 1 anh test
# (voi lop co nhieu kenh vao, trung binh theo kenh vao de ve duoc thanh anh 3x3)
# ----------------------------------------------------------------------------
def cmd_filters(a):
    dev = get_device()
    model, ck = load_run(a)
    out_dir = os.path.join(a.runs, a.name)
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    conv = convs[-1] if a.layer == "last" else convs[0]

    w = conv.weight.detach().cpu().numpy().mean(axis=1)  # (cout, 3, 3)
    n, cols = min(len(w), 64), 8
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.2))
    for i, ax in enumerate(np.atleast_1d(axes).ravel()):
        ax.axis("off")
        if i < n:
            ax.imshow(w[i], cmap="gray", interpolation="nearest")
    fig.suptitle(f"Filters cua lop Conv {a.layer} ({len(w)} filter, hien {n})")
    fig.savefig(os.path.join(out_dir, f"filters_{a.layer}.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)

    data, _ = load_data(a.data_dir)
    x, y = data["PrivateTest"]
    img = x[a.index]
    tf = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
    feats = {}
    h = conv.register_forward_hook(lambda m, i, o: feats.update(out=o.detach().cpu()))
    with torch.no_grad():
        logits = model(tf(Image.fromarray(img)).unsqueeze(0).to(dev))
    h.remove()
    fm = feats["out"][0].numpy()
    n = min(len(fm), 32)
    fig, axes = plt.subplots(4, 9, figsize=(13, 6.5))
    axes = axes.ravel()
    for ax in axes:
        ax.axis("off")
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"{ck['classes'][y[a.index]]} (that)\n{ck['classes'][logits.argmax(1).item()]} (doan)",
                      fontsize=7)
    for i in range(min(n, 35)):
        axes[i + 1].imshow(fm[i], cmap="viridis")
    fig.suptitle(f"Feature map cua lop Conv {a.layer}")
    fig.savefig(os.path.join(out_dir, f"featuremaps_{a.layer}.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("Da luu filters_*.png va featuremaps_*.png vao", out_dir)


# ----------------------------------------------------------------------------
# summary: nhat ky thuc nghiem tu dong
# ----------------------------------------------------------------------------
def cmd_summary(a):
    rows = []
    for p in sorted(glob.glob(os.path.join(a.runs, "*", "result.json"))):
        with open(p) as f:
            r = json.load(f)
        rows.append({**{k: v for k, v in r.items() if k != "config"}, **r["config"]})
    if not rows:
        print("Chua co thi nghiem nao trong", a.runs)
        return
    rows.sort(key=lambda r: -r["val_acc"])
    keys = list(rows[0])
    with open(os.path.join(a.runs, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    show = ["name", "blocks", "convs", "base", "fc", "dropout", "opt", "lr", "aug",
            "best_epoch", "val_loss", "val_acc", "test_acc"]
    print("  ".join(f"{k:>9s}" for k in show))
    for r in rows:
        print("  ".join(f"{str(r[k]):>9s}" for k in show))
    print("\nDa luu", os.path.join(a.runs, "results.csv"))


# ----------------------------------------------------------------------------
def main(argv=None):
    p = argparse.ArgumentParser(description="BTL02 - FER2013 CNN (PyTorch)")
    sub = p.add_subparsers(dest="cmd", required=True)

    def common(sp):
        sp.add_argument("--data_dir", default="data")
        sp.add_argument("--runs", default="runs")
        sp.add_argument("--workers", type=int, default=2)

    t = sub.add_parser("train", help="huan luyen 1 thi nghiem")
    common(t)
    t.add_argument("--name", required=True, help="ten thi nghiem (thu muc runs/<name>)")
    t.add_argument("--blocks", type=int, default=3, help="so khoi Conv (1-5)")
    t.add_argument("--convs", type=int, default=2, help="so lop Conv moi khoi")
    t.add_argument("--base", type=int, default=64, help="so kenh khoi dau")
    t.add_argument("--fc", type=int, default=1024, help="so no-ron lop Dense")
    t.add_argument("--dropout", type=float, default=0.2)
    t.add_argument("--bn", type=int, default=1, help="1=dung BatchNorm")
    t.add_argument("--opt", choices=["adam", "sgd"], default="adam")
    t.add_argument("--lr", type=float, default=1e-3)
    t.add_argument("--wd", type=float, default=0.0, help="weight decay")
    t.add_argument("--sched", type=int, default=1, help="1=ReduceLROnPlateau")
    t.add_argument("--aug", type=int, default=1, help="1=data augmentation")
    t.add_argument("--class_weight", type=int, default=0, help="1=trong so lop (lop it anh)")
    t.add_argument("--epochs", type=int, default=60)
    t.add_argument("--batch", type=int, default=128)
    t.add_argument("--patience", type=int, default=10, help="EarlyStopping")
    t.add_argument("--seed", type=int, default=42)
    t.set_defaults(fn=cmd_train)

    e = sub.add_parser("eval", help="danh gia 1 thi nghiem tren tap test")
    common(e)
    e.add_argument("--name", required=True)
    e.set_defaults(fn=cmd_eval)

    f = sub.add_parser("filters", help="ve filter + feature map")
    common(f)
    f.add_argument("--name", required=True)
    f.add_argument("--layer", choices=["last", "first"], default="last")
    f.add_argument("--index", type=int, default=0, help="chi so anh test de ve feature map")
    f.set_defaults(fn=cmd_filters)

    d = sub.add_parser("explore", help="kham pha du lieu (histogram, anh mau)")
    common(d)
    d.set_defaults(fn=cmd_explore)

    s = sub.add_parser("summary", help="bang tong hop cac thi nghiem")
    common(s)
    s.set_defaults(fn=cmd_summary)

    a = p.parse_args(argv)
    a.fn(a)

In [2]:
DATA_DIR = "data"   # thư mục chứa dữ liệu (fer2013.csv hoặc train/ + test/)
if os.path.isdir("/kaggle/input"):   # đang chạy trên Kaggle: tự tìm thư mục chứa train/ và test/
    for root, dirs, _ in os.walk("/kaggle/input"):
        if ("train" in dirs and "test" in dirs) or "Training" in dirs:
            DATA_DIR = root
            break
WORKERS = "2"       # nếu báo lỗi trên Windows thì đổi thành "0"
print("DATA_DIR =", DATA_DIR)

def run(cmd, *args):
    """Chạy 1 lệnh: train / eval / filters / summary."""
    main([cmd, "--data_dir", DATA_DIR, "--workers", WORKERS, *args])

from IPython.display import Image as IPImage, display
def show(name, *files):
    """Hiện ảnh kết quả đã lưu trong runs/<name>/."""
    for f in files:
        display(IPImage(filename=os.path.join("runs", name, f)))

DATA_DIR = /kaggle/input/datasets/msambare/fer2013


In [3]:
# Kiểm tra dữ liệu: thư mục đang chạy và nội dung DATA_DIR
print("Thư mục đang chạy:", os.getcwd())
print("Nội dung DATA_DIR:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "KHÔNG TỒN TẠI -> đổi DATA_DIR")

Thư mục đang chạy: /kaggle/working
Nội dung DATA_DIR: ['test', 'train']


## Cấu hình (ĐỔI Ở ĐÂY)

In [4]:
NGUOI = "An"              # đổi thành tên của bạn: An / Quyen / Thanh / Tan
CHAY_THI_NGHIEM = True    # An và Thanh: sang giai đoạn 2 (gộp kết quả) thì đổi thành False
EPOCHS = "30"             # số epoch tối đa mỗi thí nghiệm (có EarlyStopping); giảm còn 15-20 nếu muốn nhanh
BEST = None               # Thanh: để None = tự chọn model có val_acc cao nhất, hoặc ghi tên, ví dụ "large"

PHAN_CONG = {
    "An":    {"base": []},
    "Quyen": {"no_aug": ["--aug", "0"],
              "sgd":    ["--opt", "sgd", "--lr", "0.01"],
              "drop4":  ["--dropout", "0.4"],
              "cw":     ["--class_weight", "1"],
              "fail":   ["--aug", "0", "--dropout", "0", "--bn", "0"]},   # cố tình dễ overfit
    "Thanh": {"small":  ["--blocks", "2", "--convs", "2"],
              "medium": ["--blocks", "3", "--convs", "3"],
              "large":  ["--blocks", "4", "--convs", "3", "--fc", "2048"]},
    "Tan":   {},   # Tấn: khám phá dữ liệu (ô riêng bên dưới), không huấn luyện
}
assert NGUOI in PHAN_CONG, "NGUOI phải là một trong: " + ", ".join(PHAN_CONG)
print("Người chạy:", NGUOI, "| thí nghiệm:", list(PHAN_CONG[NGUOI]))

Người chạy: An | thí nghiệm: ['base']


## Chạy thử 1 epoch (bắt lỗi sớm; Thanh thử luôn phần đánh giá). Kết quả chạy thử được xoá sau đó.

In [5]:
import shutil
if CHAY_THI_NGHIEM and PHAN_CONG[NGUOI]:
    run("train", "--name", "thu", "--epochs", "1")
    if NGUOI == "Thanh":
        run("eval", "--name", "thu")
        run("filters", "--name", "thu")
        show("thu", "confusion_matrix.png", "filters_last.png")
    shutil.rmtree(os.path.join("runs", "thu"), ignore_errors=True)

Da doc lop angry: 3995 anh
Da doc lop disgust: 436 anh
Da doc lop fear: 4097 anh
Da doc lop happy: 7215 anh
Da doc lop neutral: 4965 anh
Da doc lop sad: 4830 anh
Da doc lop surprise: 3171 anh
Da doc lop angry: 958 anh
Da doc lop disgust: 111 anh
Da doc lop fear: 1024 anh
Da doc lop happy: 1774 anh
Da doc lop neutral: 1233 anh
Da doc lop sad: 1247 anh
Da doc lop surprise: 831 anh
Luu y: thu muc test (7178 anh) duoc chia ngau nhien thanh 2 nua: validation (3589) va test (3589).
Train/Val/Test: 28709 3589 3589 | lop: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] | thiet bi: cuda
So tham so: 10591431
Epoch   1 | train loss 2.0543 acc 0.2392 | val loss 1.7868 acc 0.2625 | lr 1.0e-03
KET QUA: {"name": "thu", "best_epoch": 1, "epochs_run": 1, "val_loss": 1.7868, "val_acc": 0.2625, "test_acc": 0.2658, "params": 10591431, "minutes": 0.4}


## Tấn: khám phá dữ liệu (không cần GPU)

In [6]:
if NGUOI == "Tan":
    run("explore")
    show("data_explore", "histogram.png", "samples.png")

## Chạy các thí nghiệm của bạn (thí nghiệm nào đã có kết quả thì bỏ qua)

In [7]:
if CHAY_THI_NGHIEM:
    for name, extra in PHAN_CONG[NGUOI].items():
        if os.path.exists(os.path.join("runs", name, "result.json")):
            print("Bỏ qua (đã có kết quả):", name)
            continue
        print("=" * 30, name, "=" * 30)
        try:
            run("train", "--name", name, "--epochs", EPOCHS, *extra)
        except Exception as e:   # 1 thí nghiệm lỗi không làm dừng cả lượt chạy
            print("LỖI ở thí nghiệm", name, ":", repr(e))

============================== base ==============================
Train/Val/Test: 28709 3589 3589 | lop: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] | thiet bi: cuda
So tham so: 10591431
Epoch   1 | train loss 2.0543 acc 0.2410 | val loss 1.7876 acc 0.2619 | lr 1.0e-03
Epoch   2 | train loss 1.7855 acc 0.2672 | val loss 1.7549 acc 0.2820 | lr 1.0e-03
Epoch   3 | train loss 1.7263 acc 0.2982 | val loss 1.6229 acc 0.3449 | lr 1.0e-03
Epoch   4 | train loss 1.5977 acc 0.3608 | val loss 1.4291 acc 0.4422 | lr 1.0e-03
Epoch   5 | train loss 1.5048 acc 0.4108 | val loss 1.3554 acc 0.4790 | lr 1.0e-03
Epoch   6 | train loss 1.4427 acc 0.4382 | val loss 1.4103 acc 0.4675 | lr 1.0e-03
Epoch   7 | train loss 1.4079 acc 0.4522 | val loss 1.2572 acc 0.5155 | lr 1.0e-03
Epoch   8 | train loss 1.3647 acc 0.4726 | val loss 1.2126 acc 0.5333 | lr 1.0e-03
Epoch   9 | train loss 1.3330 acc 0.4865 | val loss 1.2127 acc 0.5319 | lr 1.0e-03
Epoch  10 | train loss 1.3040 acc 0.4979 

## Nén kết quả của bạn để gửi nhóm (giai đoạn 1)

In [8]:
if CHAY_THI_NGHIEM and PHAN_CONG[NGUOI]:
    shutil.make_archive("runs_ketqua_" + NGUOI, "zip", "runs")
    print("Đã tạo runs_ketqua_" + NGUOI + ".zip (tải ở tab Output sau khi chạy xong)")
    run("summary")   # xem nhanh kết quả của riêng bạn

Đã tạo runs_ketqua_An.zip (tải ở tab Output sau khi chạy xong)
     name     blocks      convs       base         fc    dropout        opt         lr        aug  best_epoch   val_loss    val_acc   test_acc
     base          3          2         64       1024        0.2       adam      0.001          1         29     0.9893     0.6244     0.6347

Da luu runs/results.csv


## Giai đoạn 2 (An và Thanh): gộp kết quả của cả nhóm từ các file zip đã upload

In [9]:
import zipfile

def nhap_ket_qua():
    """Gom kết quả của các bạn (file runs_ketqua_*.zip hoặc thư mục đã giải nén trong /kaggle/input) vào runs/."""
    n = 0
    for root, dirs, files in os.walk("/kaggle/input"):
        if "best.pt" in files and "result.json" in files:      # 1 thí nghiệm đã giải nén sẵn
            dst = os.path.join("runs", os.path.basename(root))
            if not os.path.exists(dst):
                shutil.copytree(root, dst)
                n += 1
            dirs[:] = []
        for f in files:
            if f.endswith(".zip") and "ketqua" in f:
                with zipfile.ZipFile(os.path.join(root, f)) as z:
                    z.extractall("runs")
                n += 1
    print("Đã gộp", n, "mục. Các thí nghiệm hiện có:",
          sorted(d for d in os.listdir("runs") if os.path.exists(os.path.join("runs", d, "result.json"))))

GIAI_DOAN_2 = NGUOI in ("An", "Thanh") and not CHAY_THI_NGHIEM
if GIAI_DOAN_2:
    nhap_ket_qua()

### An: bảng tổng hợp (nhật ký thực nghiệm)

In [10]:
if GIAI_DOAN_2 and NGUOI == "An":
    run("summary")
    shutil.copy(os.path.join("runs", "results.csv"), "results.csv")   # để hiện ở tab Output

### Thanh: đánh giá model tốt nhất + vẽ filter

In [11]:
if GIAI_DOAN_2 and NGUOI == "Thanh":
    import glob, json
    results = [json.load(open(p)) for p in glob.glob("runs/*/result.json")]
    if BEST is None:
        BEST = max(results, key=lambda r: r["val_acc"])["name"]
    print("Model được đánh giá:", BEST)
    run("eval", "--name", BEST)
    run("filters", "--name", BEST)                      # filter lớp Conv cuối
    run("filters", "--name", BEST, "--layer", "first")  # filter lớp Conv đầu
    show(BEST, "curves.png", "confusion_matrix.png", "wrong_cases.png", "right_cases.png",
         "filters_last.png", "featuremaps_last.png", "filters_first.png")